# Análise Exploratória de Dados (EDA) - Sentiment140

Este notebook contém a análise exploratória completa do dataset Sentiment140 para análise de sentimentos.


## 1. Importações e Configurações


In [1]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Download de stopwords do NLTK (executar apenas uma vez)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("✓ Bibliotecas importadas com sucesso!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...


✓ Bibliotecas importadas com sucesso!


[nltk_data]   Unzipping corpora\stopwords.zip.


## 2. Carregamento dos Dados


In [ ]:
# Carregar o dataset processado
df = pd.read_csv('../data/sentiment140_processed.csv', encoding='utf-8', low_memory=False)

print(f"✓ Dataset carregado com sucesso!")
print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")


UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 232719-232720: invalid continuation byte

### 2.1 Primeiras Linhas do Dataset


In [ ]:
# Exibir as primeiras linhas
df.head(10)


### 2.2 Informações do Dataset


In [ ]:
# Informações do dataset
df.info()


## 3. Verificação de Qualidade dos Dados


### 3.1 Valores Nulos


In [ ]:
# Verificar valores nulos
print("Valores Nulos:")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")
print(f"Percentual de nulos: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.2f}%")


### 3.2 Valores Duplicados


In [ ]:
# Verificar valores duplicados
duplicados_total = df.duplicated().sum()
duplicados_texto = df.duplicated(subset=['text']).sum()

print(f"Linhas duplicadas (todas as colunas): {duplicados_total:,}")
print(f"Textos duplicados: {duplicados_texto:,}")
print(f"Percentual de textos duplicados: {(duplicados_texto / len(df) * 100):.2f}%")


## 4. Distribuição de Classes (Balanceamento)


In [ ]:
# Contagem de classes
class_counts = df['label'].value_counts().sort_index()

print("Distribuição de Classes:")
print(class_counts)
print(f"\nProporção:")
for label, count in class_counts.items():
    sentiment = "Negativo" if label == 0 else "Positivo"
    print(f"  {sentiment} (label={label}): {count:,} ({count/len(df)*100:.2f}%)")


In [ ]:
# Plotar gráfico de barras da distribuição de classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
labels = ['Negativo (0)', 'Positivo (1)']
colors = ['#e74c3c', '#2ecc71']

axes[0].bar(labels, class_counts.values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0].set_title('Distribuição de Classes', fontsize=16, fontweight='bold', pad=20)
axes[0].set_ylabel('Contagem', fontsize=12)
axes[0].set_xlabel('Sentimento', fontsize=12)
axes[0].grid(axis='y', alpha=0.3, linestyle='--')

# Adicionar valores nas barras
for i, (label, count) in enumerate(zip(labels, class_counts.values)):
    axes[0].text(i, count, f'{count:,}\n({count/len(df)*100:.1f}%)', 
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Gráfico de pizza
axes[1].pie(class_counts.values, labels=labels, autopct='%1.1f%%', 
           colors=colors, startangle=90, explode=(0.05, 0.05),
           textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Proporção de Classes', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# Conclusão sobre balanceamento
diff_percent = abs(class_counts.values[0] - class_counts.values[1]) / len(df) * 100
if diff_percent < 5:
    print(f"✓ Dataset BALANCEADO (diferença: {diff_percent:.2f}%)")
else:
    print(f"⚠ Dataset DESBALANCEADO (diferença: {diff_percent:.2f}%)")


## 5. Análise do Comprimento dos Textos


### 5.1 Criar Coluna text_length


In [ ]:
# Criar coluna com o comprimento do texto (número de caracteres)
df['text_length'] = df['text'].str.len()

print("✓ Coluna 'text_length' criada com sucesso!")
print(f"\nEstatísticas descritivas do comprimento dos textos:")
print(df['text_length'].describe())


In [ ]:
# Estatísticas por sentimento
print("\nEstatísticas de comprimento por sentimento:")
print("\nTextos NEGATIVOS (label=0):")
print(df[df['label'] == 0]['text_length'].describe())
print("\nTextos POSITIVOS (label=1):")
print(df[df['label'] == 1]['text_length'].describe())


### 5.2 Histogramas de Comprimento por Sentimento


In [ ]:
# Plotar histogramas de comprimento de texto por sentimento
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Separar dados por sentimento
negative_length = df[df['label'] == 0]['text_length']
positive_length = df[df['label'] == 1]['text_length']

# Histograma sobreposto
axes[0, 0].hist(negative_length, bins=50, alpha=0.6, label='Negativo', color='#e74c3c', edgecolor='black')
axes[0, 0].hist(positive_length, bins=50, alpha=0.6, label='Positivo', color='#2ecc71', edgecolor='black')
axes[0, 0].set_title('Distribuição de Comprimento: Sobreposto', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Comprimento do Texto (caracteres)', fontsize=11)
axes[0, 0].set_ylabel('Frequência', fontsize=11)
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(axis='y', alpha=0.3, linestyle='--')

# Histograma separado - Negativos
axes[0, 1].hist(negative_length, bins=50, color='#e74c3c', alpha=0.8, edgecolor='black')
axes[0, 1].set_title('Comprimento: Sentimento NEGATIVO', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Comprimento do Texto (caracteres)', fontsize=11)
axes[0, 1].set_ylabel('Frequência', fontsize=11)
axes[0, 1].axvline(negative_length.mean(), color='darkred', linestyle='--', linewidth=2, label=f'Média: {negative_length.mean():.1f}')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(axis='y', alpha=0.3, linestyle='--')

# Histograma separado - Positivos
axes[1, 0].hist(positive_length, bins=50, color='#2ecc71', alpha=0.8, edgecolor='black')
axes[1, 0].set_title('Comprimento: Sentimento POSITIVO', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Comprimento do Texto (caracteres)', fontsize=11)
axes[1, 0].set_ylabel('Frequência', fontsize=11)
axes[1, 0].axvline(positive_length.mean(), color='darkgreen', linestyle='--', linewidth=2, label=f'Média: {positive_length.mean():.1f}')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(axis='y', alpha=0.3, linestyle='--')

# Boxplot comparativo
box_data = [negative_length, positive_length]
bp = axes[1, 1].boxplot(box_data, labels=['Negativo', 'Positivo'], patch_artist=True,
                         notch=True, showmeans=True)
axes[1, 1].set_title('Boxplot: Comparação de Comprimento', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Comprimento do Texto (caracteres)', fontsize=11)
axes[1, 1].set_xlabel('Sentimento', fontsize=11)
axes[1, 1].grid(axis='y', alpha=0.3, linestyle='--')

# Colorir boxplots
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#2ecc71')
for patch in bp['boxes']:
    patch.set_alpha(0.6)

plt.tight_layout()
plt.show()


## 6. Word Clouds (Nuvens de Palavras)


### 6.1 Preparação dos Textos


In [ ]:
# Obter stopwords em inglês
stop_words = set(stopwords.words('english'))

# Separar textos por sentimento
negative_texts = ' '.join(df[df['label'] == 0]['text'].astype(str))
positive_texts = ' '.join(df[df['label'] == 1]['text'].astype(str))

print(f"✓ Textos preparados!")
print(f"Total de caracteres em textos negativos: {len(negative_texts):,}")
print(f"Total de caracteres em textos positivos: {len(positive_texts):,}")


### 6.2 Geração das Word Clouds


In [ ]:
# Gerar Word Clouds
print("Gerando Word Clouds... (isso pode levar alguns minutos)")

# Word Cloud para textos NEGATIVOS
wordcloud_negative = WordCloud(
    width=1600, 
    height=800,
    background_color='white',
    stopwords=stop_words,
    colormap='Reds',
    max_words=200,
    relative_scaling=0.5,
    min_font_size=10
).generate(negative_texts)

# Word Cloud para textos POSITIVOS
wordcloud_positive = WordCloud(
    width=1600, 
    height=800,
    background_color='white',
    stopwords=stop_words,
    colormap='Greens',
    max_words=200,
    relative_scaling=0.5,
    min_font_size=10
).generate(positive_texts)

print("✓ Word Clouds geradas com sucesso!")


In [ ]:
# Plotar as Word Clouds
fig, axes = plt.subplots(2, 1, figsize=(18, 14))

# Word Cloud - NEGATIVO
axes[0].imshow(wordcloud_negative, interpolation='bilinear')
axes[0].set_title('WORD CLOUD - Sentimento NEGATIVO', 
                 fontsize=20, fontweight='bold', pad=20, color='#c0392b')
axes[0].axis('off')

# Word Cloud - POSITIVO
axes[1].imshow(wordcloud_positive, interpolation='bilinear')
axes[1].set_title('WORD CLOUD - Sentimento POSITIVO', 
                 fontsize=20, fontweight='bold', pad=20, color='#27ae60')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ANÁLISE DAS WORD CLOUDS")
print("="*70)
print("\n📊 As nuvens de palavras mostram as palavras mais frequentes em cada categoria.")
print("   Quanto MAIOR a palavra, MAIS FREQUENTE ela aparece nos textos.")
print("\n🔴 NEGATIVOS: Palavras associadas a sentimentos negativos, problemas, etc.")
print("🟢 POSITIVOS: Palavras associadas a sentimentos positivos, alegria, etc.")


## 7. Exemplos de Textos


In [ ]:
# Exemplos de textos de cada classe
print("="*80)
print("EXEMPLOS DE TEXTOS NEGATIVOS (label=0)")
print("="*80)
for i, (idx, row) in enumerate(df[df['label']==0].head(5).iterrows(), 1):
    print(f"\n{i}. Texto (comprimento: {row['text_length']} caracteres):")
    print(f"   {row['text'][:200]}{'...' if len(row['text']) > 200 else ''}")

print("\n\n" + "="*80)
print("EXEMPLOS DE TEXTOS POSITIVOS (label=1)")
print("="*80)
for i, (idx, row) in enumerate(df[df['label']==1].head(5).iterrows(), 1):
    print(f"\n{i}. Texto (comprimento: {row['text_length']} caracteres):")
    print(f"   {row['text'][:200]}{'...' if len(row['text']) > 200 else ''}")


## 8. Conclusões da Análise Exploratória

### 📊 Principais Insights:

1. **Balanceamento das Classes:**
   - O dataset está balanceado/desbalanceado? (verificar gráfico acima)
   - Importante para escolha de métricas de avaliação

2. **Qualidade dos Dados:**
   - Verificação de valores nulos e duplicados
   - Necessidade de limpeza adicional

3. **Características dos Textos:**
   - Comprimento médio dos textos por sentimento
   - Textos negativos vs. positivos têm tamanhos similares?

4. **Vocabulário:**
   - Palavras-chave identificadas nas Word Clouds
   - Diferenças vocabulares entre sentimentos

### 🚀 Próximos Passos:

1. **Preprocessamento:**
   - Limpeza de textos (remoção de URLs, menções, etc.)
   - Normalização (lowercasing, remoção de pontuação)
   - Tokenização

2. **Modelagem:**
   - Modelos Baseline: Logistic Regression, Naive Bayes, SVM
   - Modelos SOTA: RoBERTa, BERT, DistilBERT

3. **Avaliação:**
   - Métricas: Accuracy, Precision, Recall, F1-Score
   - Matriz de confusão
   - Análise de erros
